In [ ]:
%%writefile database_mcp_server.py

from flask import Flask, request, jsonify
import sqlite3
from datetime import datetime

app = Flask(__name__)
DB = "mcp_orders.db"


def get_db():
    conn = sqlite3.connect(DB)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            user_id TEXT PRIMARY KEY,
            name TEXT NOT NULL
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            order_id TEXT PRIMARY KEY,
            user_id TEXT NOT NULL,
            amount REAL NOT NULL,
            order_time TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(user_id)
        )
    """)

    conn.commit()
    conn.close()


def store_order(order_id, user_id, amount):
    conn = get_db()

    conn.execute(
        """
        INSERT INTO orders (order_id, user_id, amount, order_time)
        VALUES (?, ?, ?, ?)
        """,
        (order_id, user_id, amount, datetime.now().isoformat())
    )

    conn.commit()
    conn.close()

    return {
        "success": True,
        "message": "Order stored successfully"
    }


def get_user_orders(user_id):
    conn = get_db()

    rows = conn.execute(
        """
        SELECT order_id, amount, order_time
        FROM orders
        WHERE user_id = ?
        ORDER BY order_time DESC
        """,
        (user_id,)
    ).fetchall()

    conn.close()

    return [dict(row) for row in rows]


def get_last_n_orders(user_id, n):
    conn = get_db()

    rows = conn.execute(
        """
        SELECT order_id, amount, order_time
        FROM orders
        WHERE user_id = ?
        ORDER BY order_time DESC
        LIMIT ?
        """,
        (user_id, n)
    ).fetchall()

    conn.close()

    return [dict(row) for row in rows]


def delete_order(order_id):
    conn = get_db()

    cursor = conn.execute(
        "DELETE FROM orders WHERE order_id = ?",
        (order_id,)
    )

    conn.commit()
    deleted = cursor.rowcount
    conn.close()

    if deleted:
        return {
            "success": True,
            "message": "Order deleted successfully"
        }

    return {
        "success": False,
        "message": "Order not found"
    }


@app.route("/")
def home():
    return "Database MCP Server Running"


@app.route("/command", methods=["POST"])
def command():
    data = request.get_json(silent=True) or {}
    command = data.get("command")

    try:
        if command == "ADD_ORDER":
            required = ["order_id", "user_id", "amount"]

            if not all(key in data for key in required):
                return jsonify({"error": "Missing required fields"}), 400

            result = store_order(
                data["order_id"],
                data["user_id"],
                data["amount"]
            )

            return jsonify(result)

        elif command == "GET_USER_ORDERS":
            if "user_id" not in data:
                return jsonify({"error": "user_id is required"}), 400

            return jsonify(
                get_user_orders(data["user_id"])
            )

        elif command == "GET_LAST_N_ORDERS":
            if "user_id" not in data or "n" not in data:
                return jsonify({
                    "error": "user_id and n are required"
                }), 400

            n = int(data["n"])

            if n <= 0:
                return jsonify({
                    "error": "n must be greater than 0"
                }), 400

            return jsonify(
                get_last_n_orders(data["user_id"], n)
            )

        elif command == "DELETE_ORDER":
            if "order_id" not in data:
                return jsonify({
                    "error": "order_id is required"
                }), 400

            return jsonify(
                delete_order(data["order_id"])
            )

        else:
            return jsonify({
                "error": "Unknown command"
            }), 400

    except sqlite3.IntegrityError:
        return jsonify({
            "error": "Order already exists or user does not exist"
        }), 400

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500


if __name__ == "__main__":
    init_db()
    app.run(host="127.0.0.1", port=5000, debug=False)

In [ ]:
from database_mcp_server import app, init_db

init_db()

app.run(
    host="127.0.0.1",
    port=5000,
    debug=False,
    use_reloader=False
)

In [ ]:
import sqlite3

conn = sqlite3.connect("mcp_orders.db")

conn.execute(
    "INSERT OR IGNORE INTO users (user_id, name) VALUES (?, ?)",
    ("U001", "Rahul Sharma")
)

conn.execute(
    "INSERT OR IGNORE INTO users (user_id, name) VALUES (?, ?)",
    ("U002", "Priya Patel")
)

conn.commit()
conn.close()

print("Users added successfully")

In [ ]:
import requests

url = "http://127.0.0.1:5000/command"

orders = [
    {
        "command": "ADD_ORDER",
        "order_id": "ORD001",
        "user_id": "U001",
        "amount": 450
    },
    {
        "command": "ADD_ORDER",
        "order_id": "ORD002",
        "user_id": "U001",
        "amount": 720
    },
    {
        "command": "ADD_ORDER",
        "order_id": "ORD003",
        "user_id": "U001",
        "amount": 350
    }
]

for order in orders:
    response = requests.post(url, json=order)
    print(response.json())

In [ ]:
response = requests.post(
    url,
    json={
        "command": "GET_USER_ORDERS",
        "user_id": "U001"
    }
)

print(response.json())

In [ ]:
response = requests.post(
    url,
    json={
        "command": "GET_LAST_N_ORDERS",
        "user_id": "U001",
        "n": 2
    }
)

print(response.json())

In [ ]:
def delete_order(order_id):
    response = requests.post(
        "http://127.0.0.1:5000/command",
        json={
            "command": "DELETE_ORDER",
            "order_id": order_id
        }
    )

    return response.json()


print(delete_order("ORD002"))

In [ ]:
## AI-Generated Function

I asked ChatGPT to generate a Python function that deletes an order by `order_id` from a SQLite database.

```python
def delete_order(order_id):
    conn = sqlite3.connect("mcp_orders.db")

    cursor = conn.execute(
        "DELETE FROM orders WHERE order_id = ?",
        (order_id,)
    )

    conn.commit()

    if cursor.rowcount > 0:
        result = "Order deleted successfully"
    else:
        result = "Order not found"

    conn.close()

    return result

In [ ]:
def delete_order(order_id):
    conn = get_db()

    cursor = conn.execute(
        "DELETE FROM orders WHERE order_id = ?",
        (order_id,)
    )

    conn.commit()
    deleted = cursor.rowcount
    conn.close()

    if deleted:
        return {
            "success": True,
            "message": "Order deleted successfully"
        }

    return {
        "success": False,
        "message": "Order not found"
    }